In [1]:
import numpy as np
import cvxpy as cp
import qutip as qt

In [2]:
def createKraus(dims=[5,5], params=[.1,.1],method = 'operator'):
    """Generate a channel that represents mixing the first mode (signal) with a thermal state and return the Kraus rep
    ----------
    dims: int[3], default [5,5]
        The size of the Fock spaces for respectively, the thermal modes, and the signal mode
    params: float[2], default [.1,.1]
        The transmisivity of the beamsplitter and the average photon number of the thermal field
    method: string literal, default 'operator'
        Either 'operator' or 'analytic', determines the method used to generate the thermal state as seen in qutip's thermal_dm method
    Returns
    -------
    QuTiP list[Qobj] (oper)
        The Kraus representation of a superoperator that acts on states to apply the above defined channel
    """
    eta = params[0]
    theta = np.arccos(np.sqrt(eta))
    nth = params[1]
    thdims = dims[0]
    sdims = dims[1]
    # Create our transformation, and our state to compare with
    ath = qt.tensor(qt.destroy(thdims),qt.identity(sdims))
    asi = qt.tensor(qt.identity(thdims),qt.destroy(sdims)) # can't call this as, because it's a keyword
    G = theta *1j*(ath.dag()*asi +asi.dag()*ath)
    # Define the beamsplitter transform
    U = G.expm()
    pth = qt.thermal_dm(thdims,nth,method)
    # Create a superoperator which performs the thermal mixing + tracing out
    S1=0
    for i in range(thdims):
        fockn = qt.fock(thdims,i)
        pthket = pth*fockn
        ql = qt.tensor(qt.identity(sdims),pthket).drop_scalar_dims(inplace=True)
        qr = qt.tensor(qt.identity(sdims),fockn.dag()).drop_scalar_dims(inplace=True)
        S1 = S1+qt.sprepost(ql,qr)
    S2 = qt.sprepost(U,U.dag())*S1
    S3 = qt.tensor_contract(S2,(1,3))
    # Defines a map that turns a 3 mode pure mode to a mixture of two
    # mode pure states with classical witnesses
    # Since each of these channels is a single field channel, we can
    # construct our whole channel via tensor product
    krausqm = qt.to_kraus(S3)
    krausms = [qt.dimensions.to_tensor_rep(krausqm[i]) for i in range(len(krausqm))]
    are_real = np.all(np.isreal(krausms))
    if are_real:
        return np.real(krausms)
    else:
        return krausms

In [3]:
Kraus = createKraus()

In [4]:
len(Kraus)

25

In [5]:
num = [ np.insert(np.zeros(4),i,i) for i in range(5)]

In [6]:
num

[array([0., 0., 0., 0., 0.]),
 array([0., 1., 0., 0., 0.]),
 array([0., 0., 2., 0., 0.]),
 array([0., 0., 0., 3., 0.]),
 array([0., 0., 0., 0., 4.])]

In [7]:
Krausd = np.matmul(Kraus,num)

In [8]:
Kraust = np.transpose(Kraus,[0,2,1])

In [9]:
Krausdt = np.transpose(Krausd,[0,2,1])

In [10]:
numb = [ np.insert(np.zeros(len(Kraus)-1),i,i) for i in range(len(Kraus))]

In [11]:
Ktest = np.moveaxis(np.matmul(numb,np.moveaxis(Kraus,0,1)),1,0)

In [12]:
H = cp.Variable((25,25),hermitian=True)

In [13]:
Ht = cp.conj(H)

In [14]:
Ktildep = cp.swapaxes(cp.matmul(H,np.swapaxes(Kraus,0,1)),1,0)

In [15]:
Ktilde = Krausd -1j*Ktildep

In [16]:
Ktildept = cp.swapaxes(cp.matmul(Ht,np.swapaxes(Kraust,0,1)),1,0)

In [17]:
Ktildedag = Krausdt + 1j*Ktildept

In [18]:
prods = cp.matmul(Ktildedag,Ktilde)

In [25]:
mat = cp.sum(prods,axis=0)

In [26]:
prods.is_dcp()

False

In [27]:
prob = cp.Problem(cp.Minimize(cp.lambda_max(mat)))

In [28]:
prob.solve(solver=cp.CLARABEL,verbose=True)

(CVXPY) Apr 21 03:02:50 PM: Your problem has 625 variables, 0 constraints, and 0 parameters.
(CVXPY) Apr 21 03:02:50 PM: It is compliant with the following grammars: 
(CVXPY) Apr 21 03:02:50 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 21 03:02:50 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Apr 21 03:02:50 PM: Your problem is compiled with the CPP canonicalization backend.


                                     CVXPY                                     
                                     v1.8.2                                    


DCPError: Problem does not follow DCP rules. Specifically:
The objective is not DCP. Its following subexpressions are not:
([[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
    0.00000000e+00]
  [ 7.12029603e-16  1.19012894e-17  3.26150459e-04  1.51230442e-16
    1.62000818e-15]
  [-2.49918895e-18 -5.65571367e-15 -5.91271342e-17 -7.98902776e-04
   -2.95593205e-16]
  [ 9.21357868e-19  1.33469808e-17  1.03925754e-14  4.82108423e-17
    2.58904183e-04]
  [ 0.00000000e+00 -1.83466582e-18 -1.90722616e-17 -2.98776264e-15
    4.71871114e-17]]

 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
    0.00000000e+00]
  [ 2.72442188e-04 -4.01330539e-30 -9.38127927e-15  2.78416346e-16
    4.72245199e-15]
  [ 1.28830475e-17 -2.16343740e-03  1.54242989e-29  2.28252014e-14
   -6.61338702e-16]
  [ 1.54087711e-17 -7.20005066e-17  3.97449113e-03  3.19715584e-29
   -8.37669909e-15]
  [-3.76777854e-19 -3.04987839e-17  1.08868087e-16 -1.14491738e-03
   -5.46374657e-29]]

 [